# Обучение классификатора (Этап 4)

Два режима:
1. **Random Forest** на морфометрических метриках (baseline)
2. **MLP** на latent vector + морфометрия (основной)

**Предусловие:** Нужен best_autoencoder.pt на Drive (от этапа 1).

In [ ]:
# 1. Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Скачиваем актуальный код
%cd /content
!rm -rf MultiView_3D_Prediction
!git clone https://github.com/3x6dll9ff/MultiView_3D_Prediction.git
%cd MultiView_3D_Prediction

In [ ]:
# 3. Зависимости
!pip install -r requirements.txt

In [ ]:
# 4. Подготовка: данные + базовая модель с Drive
import os, shutil

!rm -rf /content/data
!mkdir -p /content/data
!unzip -q /content/drive/MyDrive/data_processed.zip -d /content/data

DRIVE_DIR = '/content/drive/MyDrive/MultiView_Results'
base_model_drive = os.path.join(DRIVE_DIR, 'best_autoencoder.pt')
base_model_local = 'results/best_autoencoder.pt'

os.makedirs('results', exist_ok=True)
if os.path.exists(base_model_drive):
    shutil.copy2(base_model_drive, base_model_local)
    print(f"Base model скопирован")
else:
    print("ОШИБКА: best_autoencoder.pt не найден на Drive!")

In [ ]:
# 5a. Random Forest (baseline — быстро)
!DATA_PATH=$(find /content/data -name "metadata.csv" | head -n 1 | xargs dirname) && \
 python3 src/train_classifier.py \
    --data_dir "$DATA_PATH" \
    --mode rf

In [ ]:
# 5b. MLP классификатор на latent + морфометрия
!DATA_PATH=$(find /content/data -name "metadata.csv" | head -n 1 | xargs dirname) && \
 python3 src/train_classifier.py \
    --data_dir "$DATA_PATH" \
    --mode latent \
    --autoencoder results/best_autoencoder.pt \
    --input_mode tri \
    --epochs 40 \
    --batch_size 16 \
    --lr 1e-3

In [ ]:
# 6. Сохраняем результаты на Drive
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/MultiView_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)

for f in ['best_classifier.pt']:
    src = os.path.join('results', f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_DIR, f))
        print(f"Сохранено: {f}")

metrics_dir = os.path.join(DRIVE_DIR, 'metrics')
os.makedirs(metrics_dir, exist_ok=True)
for f in os.listdir('results/metrics'):
    src = os.path.join('results/metrics', f)
    if ('rf' in f or 'classifier' in f) and os.path.isfile(src):
        shutil.copy2(src, os.path.join(metrics_dir, f))
        print(f"Сохранено: metrics/{f}")

print(f"\nКлассификатор на Drive: {DRIVE_DIR}/best_classifier.pt")